A trained CLIP model can classify images into categories it was never trained on, without a single labeled example. This section shows that the procedure is an ordinary linear classifier whose weights are written in English, measures it on the 10,000 test images of CIFAR-10, and compares it with a classifier trained on labeled examples. The notation follows the [CLIP section](/aiml-common/lectures/vlm/clip/index): $\boldsymbol\ell$ is an image, $\mathbf{t}$ a text, $f_\ell$ and $f_t$ the two encoders, and $\mathbf{z}_\ell = f_\ell(\boldsymbol\ell)$ and $\mathbf{z}_t = f_t(\mathbf{t})$ their unit-length embeddings.

## Zero-shot classification is a linear classifier

For classes $y \in \{1, \dots, K\}$, write a prompt $\mathbf{t}^y$ for each class, such as "a photo of a dog.", and embed it, $\mathbf{w}_y = f_t(\mathbf{t}^y)$. For an image with embedding $\mathbf{z}_\ell$, predict the class whose prompt is most similar:

$$
\hat{y} = \arg\max_y \; \mathbf{w}_y^\top \mathbf{z}_\ell .
$$

This is a linear classifier on the image embedding. The weight matrix $W = [\mathbf{w}_1, \dots, \mathbf{w}_K]^\top$ has one row per class, and the scores $W \mathbf{z}_\ell$ followed by a softmax give class probabilities, exactly as in [logistic regression](/aiml-common/lectures/classification/logistic-regression/index). The difference is where the weights come from. Logistic regression fits $W$ to labeled images. Zero-shot classification computes $W$ from text, so changing the classes means changing the prompts, and no image of the new classes is needed. The quality of the classifier then depends on two things: how well the image embedding separates the classes, and how well the prompts point at them.

In [ ]:
import contextlib
import io
import os

import matplotlib.pyplot as plt
import transformers
from torchvision import datasets as _tv_datasets

plt.rcParams["figure.dpi"] = 110
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
transformers.utils.logging.disable_progress_bar()
transformers.utils.logging.set_verbosity_error()
with contextlib.redirect_stderr(io.StringIO()), contextlib.redirect_stdout(io.StringIO()):
    _tv_datasets.CIFAR10("data", train=True, download=True)

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets
from transformers import AutoModel, AutoProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)

model = AutoModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
processor = AutoProcessor.from_pretrained("openai/clip-vit-base-patch32")
print(f"[INFO] device: {device}")

## Embedding the data

The image encoder runs once over the CIFAR-10 test set and once over the training set. Every classifier below, zero-shot or trained, works on these frozen embeddings. The images are $32 \times 32$ and are enlarged to $224 \times 224$ by the processor, which makes this a harder test than the photos CLIP was trained on.

In [ ]:
@torch.no_grad()
def embed_images(dataset, batch_size=500):
    loader = DataLoader(dataset, batch_size=batch_size, collate_fn=lambda b: b, num_workers=4)
    feats, labels = [], []
    for batch in loader:
        x = processor(images=[im for im, _ in batch], return_tensors="pt").to(device)
        feats.append(F.normalize(model.get_image_features(**x).pooler_output, dim=-1).float().cpu())
        labels += [y for _, y in batch]
    return torch.cat(feats), torch.tensor(labels)


@torch.no_grad()
def embed_texts(texts):
    x = processor(text=texts, return_tensors="pt", padding=True).to(device)
    return F.normalize(model.get_text_features(**x).pooler_output, dim=-1).float().cpu()


train_set = datasets.CIFAR10("data", train=True, download=True)
test_set = datasets.CIFAR10("data", train=False, download=True)
CLASSES = test_set.classes
Z_test, y_test = embed_images(test_set)
Z_train, y_train = embed_images(train_set)
print(f"[INFO] test embeddings {tuple(Z_test.shape)}, training embeddings {tuple(Z_train.shape)}")

## Prompts

Radford et al. found that the bare class name is a weak prompt [@Radford2021-hl]. Captions on the web are sentences, not single words, and a single word can be ambiguous: "crane" is a bird and a machine. Wrapping the name in a sentence such as `a photo of a {class}.` moves the prompt closer to the captions the text encoder was trained on. Averaging the embeddings of several such sentences, *prompt ensembling*, gives a weight vector that depends less on the wording of any one of them. The cell compares the three choices.

In [ ]:
TEMPLATES = [
    "a photo of a {}.",
    "a blurry photo of a {}.",
    "a low resolution photo of a {}.",
    "a pixelated photo of a {}.",
    "a photo of the small {}.",
    "a photo of the big {}.",
    "a black and white photo of a {}.",
    "a drawing of a {}.",
]


def accuracy(W, Z=Z_test, y=y_test):
    return ((Z @ W.T).argmax(1) == y).float().mean().item()


W_name = embed_texts(CLASSES)
W_photo = embed_texts([f"a photo of a {c}." for c in CLASSES])
W_ensemble = F.normalize(
    torch.stack([embed_texts([t.format(c) for t in TEMPLATES]).mean(0) for c in CLASSES]), dim=-1
)
for name, W in [("class name", W_name), ('"a photo of a {class}."', W_photo),
                (f"ensemble of {len(TEMPLATES)} templates", W_ensemble)]:
    print(f"{name:<32} test accuracy {accuracy(W):.3f}")

PROMPT_PLACEHOLDER

In [ ]:
pred = (Z_test @ W_ensemble.T).argmax(1)
confusion = torch.zeros(10, 10)
for t, p in zip(y_test, pred):
    confusion[t, p] += 1
confusion /= confusion.sum(1, keepdim=True)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(confusion, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(10))
ax.set_xticklabels(CLASSES, rotation=45, ha="right")
ax.set_yticks(range(10))
ax.set_yticklabels(CLASSES)
ax.set_xlabel("predicted")
ax.set_ylabel("true")
for i in range(10):
    for j in range(10):
        if confusion[i, j] >= 0.05:
            ax.text(j, i, f"{confusion[i, j]:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if confusion[i, j] > 0.5 else "black")
ax.set_title("Zero-shot confusion matrix, prompt ensemble")
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

CONFUSION_PLACEHOLDER

## Zero-shot against a linear probe

How good is a classifier written in English, compared with one trained on labeled images? A *linear probe* answers this on the same embedding. It trains a logistic regression on the frozen image embeddings of $k$ labeled images per class and is tested on the same 10,000 test images. Both classifiers are linear in $\mathbf{z}_\ell$, so the comparison measures only where the weights come from. Radford et al. found that, averaged over 27 data sets, zero-shot CLIP matched a linear probe trained with 4 examples per class.

In [ ]:
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(0)
by_class = [np.flatnonzero(y_train.numpy() == c) for c in range(10)]
shots = [1, 2, 4, 8, 16, 64, 256]
probe = {}
for k in shots:
    accs = []
    for seed in range(5 if k <= 16 else 1):  # average the small-k runs over 5 draws
        r = np.random.default_rng(seed)
        idx = np.concatenate([r.choice(ix, k, replace=False) for ix in by_class])
        clf = LogisticRegression(max_iter=2000, C=10.0).fit(Z_train[idx], y_train[idx])
        accs.append(clf.score(Z_test, y_test))
    probe[k] = float(np.mean(accs))
    print(f"linear probe, {k:>3} examples per class: test accuracy {probe[k]:.3f}")

full = LogisticRegression(max_iter=3000, C=10.0).fit(Z_train, y_train)
probe_full = full.score(Z_test, y_test)
print(f"linear probe, all 5,000 per class:   test accuracy {probe_full:.3f}")
print(f"zero-shot, prompt ensemble:          test accuracy {accuracy(W_ensemble):.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(shots, [probe[k] for k in shots], "o-", label="linear probe")
ax.axhline(accuracy(W_ensemble), color="C1", ls="--", label="zero-shot (prompt ensemble)")
ax.axhline(probe_full, color="gray", ls=":", label="linear probe, all 5,000 per class")
ax.set_xscale("log", base=2)
ax.set_xticks(shots)
ax.set_xticklabels(shots)
ax.set_xlabel("labeled examples per class")
ax.set_ylabel("test accuracy")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

PROBE_PLACEHOLDER

## Exercises

1. Add the eight templates one at a time, in the order given, and plot the accuracy against the number of templates.
2. Replace "automobile" with "car" and "airplane" with "plane" in the prompts. Does the accuracy change? What does this tell you about choosing class names?
3. Initialize the weights of the logistic regression with the zero-shot weights $W$ (scaled by 100) and train it with 4 examples per class. Does starting from the text help?
4. Evaluate the zero-shot classifier on CIFAR-100, which has 100 classes. How does the gap to the linear probe change with the number of classes?

## Key references

Radford et al. introduced zero-shot transfer with CLIP and compared it with linear probes on 27 data sets [@Radford2021-hl].